In [24]:
import os
import requests
import time

FLINK_URL = f"http://flink.{os.environ.get('INGRESS_IP', '127.0.0.1')}.sslip.io"
PUSHGATEWAY_URL = f"http://pushgateway.{os.environ.get('INGRESS_IP', '127.0.0.1')}.sslip.io"


In [25]:
def push_managed_memory():
    """Fetch allocated managed memory from /jobs/{id}/justin and push to Pushgateway."""
    try:
        jobs = requests.get(f"{FLINK_URL}/jobs", timeout=5).json().get("jobs", [])
        running = [j["id"] for j in jobs if j["status"] == "RUNNING"]
        if not running:
            return
        job_id = running[0]

        data = requests.get(f"{FLINK_URL}/jobs/{job_id}/justin", timeout=5).json()

        lines = []
        for op_id, op in data.items():
            mem = op["resourceProfile"]["managedMemory"]["bytes"]
            par = op["parallelism"]["upperBound"]
            lines.append(f'flink_managed_memory_allocated_bytes{{operator_id="{op_id}"}} {mem * par}')

        total = sum(
            op["resourceProfile"]["managedMemory"]["bytes"] * op["parallelism"]["upperBound"]
            for op in data.values()
        )
        lines.append(f"flink_managed_memory_allocated_bytes_total {total}")

        requests.post(
            f"{PUSHGATEWAY_URL}/metrics/job/flink",
            data="\n".join(lines) + "\n",
            headers={"Content-Type": "text/plain"},
            timeout=5,
        )
    except Exception as e:
        print(repr(e))
        pass


def poll_managed_memory(duration, interval=30):
    """Block for `duration` seconds, pushing managed memory metrics every `interval` seconds."""
    end = time.time() + duration
    while time.time() < end:
        push_managed_memory()
        time.sleep(min(interval, max(0, end - time.time())))


def wait_for_job(timeout=300, interval=10):
    """Block until a Flink job is RUNNING, then return. Raises TimeoutError if timeout exceeded."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            jobs = requests.get(f"{FLINK_URL}/jobs", timeout=5).json().get("jobs", [])
            if any(j["status"] == "RUNNING" for j in jobs):
                return
        except Exception:
            pass
        time.sleep(interval)
    raise TimeoutError(f"No Flink job reached RUNNING state within {timeout}s")


In [26]:
xp_duration = 600 # Time in second

# Observing Throughputs

We can use Grafana to access Prometheus metrics and observe the throughput reached for each configuration.

From a terminal, we need to expose the service to the outside using the `port-forward` utility.
```bash
$ kubectl port-forward -n manager svc/prom-grafana 3000:80
```
Grafana is now available on port 3000.

[http://localhost:3000/explore?orgId=1&left=%5B%22now-3h%22,%22now%22,%22Prometheus%22,%7B%22refId%22:%22A%22,%22instant%22:true,%22range%22:true,%22exemplar%22:true,%22expr%22:%22sum(flink_taskmanager_job_task_operator_numRecordsOutPerSecond%7Boperator_name%3D~%5C%22Source.*%5C%22%7D)%22%7D%5D](http://localhost:3000/explore?orgId=1&left=%5B%22now-3h%22,%22now%22,%22Prometheus%22,%7B%22refId%22:%22A%22,%22instant%22:true,%22range%22:true,%22exemplar%22:true,%22expr%22:%22sum(flink_taskmanager_job_task_operator_numRecordsOutPerSecond%7Boperator_name%3D~%5C%22Source.*%5C%22%7D)%22%7D%5D)

Credentials are: 
- admin
- prom-operator

Flink doesn't expose the amount of managed memory allocated. This amount needs to be manually computed by either looking at the Slot Allocation in the JobManager's logs, or by looking at the Scaling Configuration (c.f. Section 4.1 of the paper) in the Operator's logs
You can find the Scaling Configuration applied by using the following command:
```bash
$ kubectl get pods   --no-headers=true | awk '{print $1}' | grep flink-kubernetes-operator | xargs kubectl logs | grep ScalingConfiguration
2025-03-14 14:14:30,481 o.a.f.a.ScalingExecutor        [INFO ][default/flink] ScalingConfigurations{currentPeriod={}, scalingConfiguration={df21eaa579f0fa28512847e1d151ffe9={0=ScalingConfiguration{scaling={afcfe443e23d73c4737dc3635c2866c0=ScalingInformation{avgThroughput=599954.01, parallelism=3, memoryLevel=-1, verticalScaling=false, horizontalScaling=false, avgCacheHitRate=0.0, avgStateLatency=0.0}, d0fdb9c6859d778b65cf8bc10eeef578=ScalingInformation{avgThroughput=Infinity, parallelism=15, memoryLevel=-1, verticalScaling=false, horizontalScaling=false, avgCacheHitRate=0.0, avgStateLatency=0.0}, 4fe84db7e190cf467a7cebccc5b77d31=ScalingInformation{avgThroughput=5568026.319, parallelism=1, memoryLevel=-1, verticalScaling=false, horizontalScaling=false, avgCacheHitRate=0.0, avgStateLatency=0.0}}}}}}
```

In the logs: `df21eaa579f0fa28512847e1d151ffe9` is the job ID, `0`is the time period, and `afcfe443e23d73c4737dc3635c2866c0`, `d0fdb9c6859d778b65cf8bc10eeef578`, and `4fe84db7e190cf467a7cebccc5b77d31` are the operators of the job.

# Nexmark

### Query 1

In [5]:
!sed -e "s/JUSTIN/false/g" q1/query1.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q1/query1.yaml
time.sleep(30)
!sed -e "s/JUSTIN/true/g" q1/query1.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q1/query1.yaml

flinkdeployment.flink.apache.org "flink" deleted


### Query 2

In [10]:
!sed -e "s/JUSTIN/false/g" q2/query2.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q2/query2.yaml
time.sleep(30)
!sed -e "s/JUSTIN/true/g" q2/query2.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q2/query2.yaml

flinkdeployment.flink.apache.org/flink created
flinkdeployment.flink.apache.org "flink" deleted
flinkdeployment.flink.apache.org/flink created
flinkdeployment.flink.apache.org "flink" deleted


### Query 3

In [ ]:
# Vanilla run
!sed -e "s/JUSTIN/false/g" q3/query3.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q3/query3.yaml
time.sleep(30)
# Justin run
!sed -e "s/JUSTIN/true/g" q3/query3.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q3/query3.yaml

### Query 5

In [3]:
# Vanilla run
!sed -e "s/JUSTIN/false/g" q5/query5.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q5/query5.yaml
time.sleep(30)
# Justin run
!sed -e "s/JUSTIN/true/g" q5/query5.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q5/query5.yaml

flinkdeployment.flink.apache.org/flink created
flinkdeployment.flink.apache.org "flink" deleted
flinkdeployment.flink.apache.org/flink created
flinkdeployment.flink.apache.org "flink" deleted


### Query 8

In [ ]:
# Vanilla run
!sed -e "s/JUSTIN/false/g" q8/query8.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration * 2)
!kubectl delete -f q8/query8.yaml
time.sleep(30)
# Justin run
!sed -e "s/JUSTIN/true/g" q8/query8.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration * 2)
!kubectl delete -f q8/query8.yaml

flinkdeployment.flink.apache.org/flink unchanged


### Query 11

In [ ]:
# Vanilla run
!sed -e "s/JUSTIN/false/g" q11/query11.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q11/query11.yaml
time.sleep(30)
# Justin run
!sed -e "s/JUSTIN/true/g" q11/query11.yaml |  kubectl apply -f -
wait_for_job()
poll_managed_memory(xp_duration)
!kubectl delete -f q11/query11.yaml